# Appartments in Prague - price prediction

finaleeeee


nejlepsi verze




tady bez clusteringu


In [37]:
import pandas as pd
import numpy as np
import re
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, GradientBoostingRegressor, VotingRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

# ==============================================================================
# ⚙️ MASTER KONFIGURACE
# ==============================================================================

# 1. RYCHLOSTI UČENÍ (Testujeme 3 varianty)
LEARNING_RATES = [0.055, 0.05, 0.01]

# 2. EARLY STOPPING (Záchranná brzda)
# Model může běžet až 10 000 kroků, ale pokud se 50x nezlepší, vypne se.
MAX_ITER = 10000
PATIENCE = 400

# 3. ODSTRANĚNÍ ODPADU (OUTLIERS)
# Kolik procent extrémů smazat? 0.005 = smažeme jen 0.5 % nejlevnějších a 0.5 % nejdražších.
# Je to velmi jemné nastavení, aby se smazaly jen chyby, ne drahé vily.
OUTLIER_THRESHOLD = 0.005

# 4. VÁHY PRO TÝM (TRIO)
# [3, 1, 1] = HistGradient má 3 hlasy, GradientBoosting 1, RandomForest 1
#TRIO_WEIGHTS = [3, 1, 1] dava nejlepsi vysledky zatim 9,16
#TRIO_WEIGHTS = [5, 2, 1] dava nejlepsi vysledky zatim jeste lepsii 9,06
TRIO_WEIGHTS = [5, 2, 1]

# ==============================================================================
# 1. ODSTRANĚNÍ OUTLIERŮ (Jemné čištění)
# ==============================================================================
def remove_outliers(df, threshold):
    df_clean = df.copy()

    # Cena za m2 je nejlepší detektor
    df_clean['price_m2'] = df_clean['price'] / df_clean['area']

    # Hranice
    low_limit = df_clean['price_m2'].quantile(threshold)
    high_limit = df_clean['price_m2'].quantile(1 - threshold)

    # Filtrace
    initial_count = len(df_clean)
    df_clean = df_clean[
        (df_clean['price_m2'] >= low_limit) &
        (df_clean['price_m2'] <= high_limit)
    ]
    final_count = len(df_clean)

    print(f"🗑️ OČISTA: Smazáno {initial_count - final_count} silných outlierů (Práh: {threshold*100}%).")
    return df_clean

# ==============================================================================
# 2. FEATURE ENGINEERING (Agresivní Text Mining)
# ==============================================================================
def enhance_data(df):
    df = df.copy()

    # A) Dispozice
    def get_rooms(val):
        if not isinstance(val, str): return 1
        match = re.search(r'\d+', val)
        if match: return int(match.group())
        return 1
    df['n_rooms'] = df['layout'].apply(get_rooms)

    # B) Text Mining (Slovník 7.0)
    df['text_lower'] = df['text'].astype(str).str.lower()
    keywords = {
        'txt_metro': ['metro', 'metra', 'metru'],
        'txt_park': ['park', 'stromovk', 'letn', 'riegr', 'vítkov'],
        'txt_water': ['vltav', 'nábřež', 'výhled na vodu'],
        'txt_reconstructed': ['rekonstru', 'po oprav', 'nové jádro', 'zděné jádro', 'zdařil'],
        'txt_new': ['novostav', 'nový byt', 'projekt', 'kolaudac', 'developersk'],
        'txt_brick': ['cihl', 'činžovní', 'skelet'],
        'txt_balcony': ['balk', 'lodž', 'teras', 'zahrádk', 'předzahr'],
        'txt_parking': ['parkov', 'garáž', 'stání'],
        'txt_lift': ['výtah'],
        'txt_ac': ['klimatiza', 'klima'],
        'txt_storage': ['komor', 'šatn', 'sklep'],
        'txt_luxury': ['luxus', 'nadstandard', 'design', 'reziden'],  #'atypic' zkousim vyhodit
        'txt_view': ['výhled', 'panoram', 'světlý', 'slunný'],
        'txt_high_ceilings': ['vysoké stropy', 'vysokými stropy'],
        'txt_bad_condition': ['původ', 'umakart', 'před rekonstru', 'k opravě'],
        'txt_low_floor': ['přízemí', 'suterén'],
        'txt_busy': ['rušná', 'hlučn'],
        'txt_coop': ['družst']
    }
    for col, terms in keywords.items():
        pattern = '|'.join(terms)
        df[col] = df['text_lower'].str.contains(pattern, regex=True).astype(int)
    df = df.drop(columns=['text_lower'])

    # C) Čtvrť & GPS
    def get_district(addr):
        if not isinstance(addr, str) or '-' not in addr: return "Unknown"
        return addr.split('-')[-1].strip()
    df['district'] = df['address'].apply(get_district)

    # GPS Vzdálenost
    center_lat, center_lon = 50.079, 14.430
    df['dist_center'] = np.sqrt((df['gps_lat'] - center_lat)**2 + (df['gps_lon'] - center_lon)**2)

    # D) Nuly
    zero_fill_cols = ["garden_area", "balcony_area", "cellar_area", "parking"]
    for col in zero_fill_cols:
        if col in df.columns:
            if col == 'parking': df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
            else: df[col] = df[col].fillna(0)
    return df

# ==============================================================================
# 3. PŘÍPRAVA DAT
# ==============================================================================

# 1. Čištění (Jen train data!)
print("Aplikuji čištění outlierů...")
train_clean = remove_outliers(train, OUTLIER_THRESHOLD)

# 2. Feature Engineering
print("Vylepšuji data (Text Mining)...")
train_enhanced = enhance_data(train_clean)
test_enhanced  = enhance_data(test)

y = np.log1p(train_clean["price"])
drop_cols = ["price", "id", "address", "text", "first_seen", "last_seen", "layout", "price_m2"]
X = train_enhanced.drop(columns=drop_cols, errors='ignore')
X_test_final = test_enhanced.drop(columns=[c for c in drop_cols if c != "price"], errors='ignore')

# Preprocessing
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
if 'district' not in categorical_cols: categorical_cols.append('district')

preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=2))
    ]), categorical_cols)
])

# ==============================================================================
# 4. MEGA TURNAJ (S výpisem iterací)
# ==============================================================================

#X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
results_list = []
best_mape = float("inf")
best_pipe = None
best_desc = ""

print(f"\n{'='*85}")
print(f"🏎️ STARTUJI TURNAJ (Sledujeme Early Stopping)")
print(f"   Max Iter: {MAX_ITER} | Patience: {PATIENCE}")
print(f"{'='*85}")

for lr in LEARNING_RATES:

    # --- 1. Solo HGB (S brzdou) ---
    hgb = HistGradientBoostingRegressor(
        learning_rate=lr,
        max_iter=MAX_ITER,
        max_leaf_nodes=45,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=PATIENCE,
        random_state=42
    )

    # --- 2. Trio Team (Brzda i u GB!) ---
    gb = GradientBoostingRegressor(
        learning_rate=lr,
        n_estimators=3000,  # Limit
        max_depth=4,
        n_iter_no_change=PATIENCE, # <--- Brzda pro GB
        validation_fraction=0.1,
        random_state=42
    )
    rf = RandomForestRegressor(n_estimators=300, n_jobs=-1, random_state=42)

    trio = VotingRegressor(
        estimators=[('hgb', hgb), ('gb', gb), ('rf', rf)],
        weights=TRIO_WEIGHTS
    )

    current_candidates = {f"Solo HGB (lr={lr})": hgb, f"Trio Team (lr={lr})": trio}

    for name, model in current_candidates.items():
        pipe = Pipeline([("prep", preprocessor), ("model", model)])

        print(f"Trénuji: {name:<25} ... ", end="")
        pipe.fit(X_train, y_train)

        # --- Špionáž: Zjistíme, kdy zastavil ---
        stop_msg = ""
        try:
            if "Solo HGB" in name:
                n_iter = pipe.named_steps['model'].n_iter_
                stop_msg = f"[Stop: {n_iter}]"
            elif "Trio" in name:
                hgb_iter = pipe.named_steps['model'].estimators_[0].n_iter_
                gb_iter = pipe.named_steps['model'].estimators_[1].n_estimators_
                stop_msg = f"[HGB: {hgb_iter}, GB: {gb_iter}]"
        except:
            stop_msg = "[?]"

        # Výpočet MAPE
        y_pred = np.expm1(pipe.predict(X_val))
        mape = mean_absolute_percentage_error(np.expm1(y_val), y_pred) * 100

        print(f"MAPE: {mape:.2f} %  {stop_msg}")

        results_list.append({'name': name, 'mape': mape, 'stop_info': stop_msg})

        if mape < best_mape:
            best_mape = mape
            best_pipe = pipe
            best_desc = name

# ==============================================================================
# 5. FINÁLNÍ ŽEBŘÍČEK A EXPORT
# ==============================================================================
print(f"\n\n{'='*85}")
print(f"📊 KONEČNÝ ŽEBŘÍČEK MODELŮ")
print(f"{'='*85}")
print(f"{'Pořadí':<8} | {'Model':<25} | {'MAPE':<10} | {'Iterace (Stop)'}")
print("-" * 85)

results_list.sort(key=lambda x: x['mape'])

for i, res in enumerate(results_list):
    print(f"{i+1}. místo | {res['name']:<25} | {res['mape']:.2f} %   | {res['stop_info']}")

final_est = best_mape - 0.05
print("-" * 85)
print(f"\n🏆 VÍTĚZ: {best_desc}")
print(f"🔸 ODHAD PO DOUČENÍ NA 100% DATECH: {final_est:.2f} %")

print(f"\nGeneruji finální soubor vítěze (Training on Full Data)...")
best_pipe.fit(X, y)

submission = pd.DataFrame({
    "id": test["id"],
    "price": np.expm1(best_pipe.predict(X_test_final))
})

filename = "submission_final_master.csv"
submission.to_csv(filename, index=False)
print(f"🎉 Hotovo! Soubor '{filename}' je připraven.")

Aplikuji čištění outlierů...
🗑️ OČISTA: Smazáno 50 silných outlierů (Práh: 0.5%).
Vylepšuji data (Text Mining)...

🏎️ STARTUJI TURNAJ (Sledujeme Early Stopping)
   Max Iter: 10000 | Patience: 400
Trénuji: Solo HGB (lr=0.055)       ... MAPE: 9.23 %  [Stop: 1359]
Trénuji: Trio Team (lr=0.055)      ... MAPE: 9.04 %  [HGB: 1359, GB: 1091]
Trénuji: Solo HGB (lr=0.05)        ... MAPE: 9.23 %  [Stop: 2734]
Trénuji: Trio Team (lr=0.05)       ... MAPE: 9.03 %  [HGB: 2734, GB: 1283]
Trénuji: Solo HGB (lr=0.01)        ... MAPE: 9.21 %  [Stop: 3864]
Trénuji: Trio Team (lr=0.01)       ... MAPE: 9.14 %  [HGB: 3864, GB: 2071]


📊 KONEČNÝ ŽEBŘÍČEK MODELŮ
Pořadí   | Model                     | MAPE       | Iterace (Stop)
-------------------------------------------------------------------------------------
1. místo | Trio Team (lr=0.05)       | 9.03 %   | [HGB: 2734, GB: 1283]
2. místo | Trio Team (lr=0.055)      | 9.04 %   | [HGB: 1359, GB: 1091]
3. místo | Trio Team (lr=0.01)       | 9.14 %   | [HGB: 

finalni vyherce model natrenovani a odevzdani

In [40]:
print("\n==============================================================")
print("🏁 TEST OVERFITTINGU — 5 náhodných splitů")
print("==============================================================")

from sklearn.model_selection import train_test_split

N_RUNS = 5
LR = 0.05
TRIO_WEIGHTS = [5, 2, 1]

scores = []

for i in range(1, N_RUNS + 1):
    print(f"\n🔄 Běh {i}/{N_RUNS}")

    # Random split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=49 + i
    )

    # --- MODELS ---
    hgb = HistGradientBoostingRegressor(
        learning_rate=LR,
        max_iter=MAX_ITER,
        max_leaf_nodes=45,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=PATIENCE,
        random_state=49 + i
    )

    gb = GradientBoostingRegressor(
        learning_rate=LR,
        n_estimators=3000,
        max_depth=4,
        n_iter_no_change=PATIENCE,
        validation_fraction=0.1,
        random_state=49 + i
    )

    rf = RandomForestRegressor(
        n_estimators=300,
        n_jobs=-1,
        random_state=49 + i
    )

    trio = VotingRegressor(
        estimators=[('hgb', hgb), ('gb', gb), ('rf', rf)],
        weights=TRIO_WEIGHTS
    )

    # Pipeline
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", trio)
    ])

    print("   📚 Trénuji model...")
    pipe.fit(X_train, y_train)

    # Predict + compute MAPE
    y_pred = np.expm1(pipe.predict(X_val))
    mape = mean_absolute_percentage_error(np.expm1(y_val), y_pred) * 100

    scores.append(mape)
    print(f"   ✅ MAPE Split {i}: {mape:.2f} %")

# =============================================
# VÝSLEDKY
# =============================================
print("\n==============================================================")
print("📊 VÝSLEDKY 5 SPLITŮ")
print("==============================================================")

for i, score in enumerate(scores):
    print(f"Běh {i+1}: {score:.2f} %")

print("--------------------------------------------------------------")
print(f"🏆 PRŮMĚR: {np.mean(scores):.2f} %")
print("==============================================================")



🏁 TEST OVERFITTINGU — 5 náhodných splitů

🔄 Běh 1/5
   📚 Trénuji model...
   ✅ MAPE Split 1: 9.04 %

🔄 Běh 2/5
   📚 Trénuji model...
   ✅ MAPE Split 2: 9.42 %

🔄 Běh 3/5
   📚 Trénuji model...
   ✅ MAPE Split 3: 9.17 %

🔄 Běh 4/5
   📚 Trénuji model...
   ✅ MAPE Split 4: 9.61 %

🔄 Běh 5/5
   📚 Trénuji model...
   ✅ MAPE Split 5: 8.80 %

📊 VÝSLEDKY 5 SPLITŮ
Běh 1: 9.04 %
Běh 2: 9.42 %
Běh 3: 9.17 %
Běh 4: 9.61 %
Běh 5: 8.80 %
--------------------------------------------------------------
🏆 PRŮMĚR: 9.21 %


novi feturky
